# Step 1: Corpus Acquisition & F0/F1 Conversion

Abraham Tedla (wqp7qy@virginia.edu)  
DS 5001  
April 2026

Download selected novels from Project Gutenberg, strip boilerplate, parse into MLCF (F1), and build the LIBRARY table.

# Set Up

In [59]:
import sqlite3
import pandas as pd
import numpy as np
import os
import re
import requests
import time

np.random.seed(42)

In [60]:
import seaborn as sns
sns.set()
%matplotlib inline

In [61]:
db_path = 'gutenberg.db'
source_dir = 'source_texts'
output_dir = 'output'

# OHCO: Ordered Hierarchy of Content Objects
# For novels: book_id > chapter > para_num > sent_num
OHCO = ['book_id', 'chapter', 'para_num', 'sent_num']
BOOKS = OHCO[:1]
CHAPS = OHCO[:2]
PARAS = OHCO[:3]
SENTS = OHCO[:4]

# Selected corpus: 12 novels across Gothic/Horror and Science Fiction
CORPUS = {
    84:  {'title': 'Frankenstein', 'author': 'Shelley, Mary', 'genre': 'Gothic'},
    345: {'title': 'Dracula', 'author': 'Stoker, Bram', 'genre': 'Gothic'},
    42:  {'title': 'Dr. Jekyll and Mr. Hyde', 'author': 'Stevenson, Robert Louis', 'genre': 'Gothic'},
    768: {'title': 'Wuthering Heights', 'author': 'Brontë, Emily', 'genre': 'Gothic'},
    174: {'title': 'The Picture of Dorian Gray', 'author': 'Wilde, Oscar', 'genre': 'Gothic'},
    696: {'title': 'The Castle of Otranto', 'author': 'Walpole, Horace', 'genre': 'Gothic'},
    35:  {'title': 'The Time Machine', 'author': 'Wells, H. G.', 'genre': 'SciFi'},
    36:  {'title': 'The War of the Worlds', 'author': 'Wells, H. G.', 'genre': 'SciFi'},
    159: {'title': 'The Island of Doctor Moreau', 'author': 'Wells, H. G.', 'genre': 'SciFi'},
    164: {'title': 'Twenty Thousand Leagues Under the Sea', 'author': 'Verne, Jules', 'genre': 'SciFi'},
    62:  {'title': 'A Princess of Mars', 'author': 'Burroughs, Edgar Rice', 'genre': 'SciFi'},
    139: {'title': 'The Lost World', 'author': 'Doyle, Arthur Conan', 'genre': 'SciFi'},
}

In [62]:
os.makedirs(source_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Download Source Texts (F0)

Gutenberg plaintext URLs follow the pattern: `https://www.gutenberg.org/cache/epub/{gid}/pg{gid}.txt`

In [63]:
def download_text(gid, dest_dir):
    """Download a Gutenberg plaintext file by its ID."""
    filepath = os.path.join(dest_dir, f'{gid}.txt')
    if os.path.exists(filepath):
        print(f'  {gid}: already downloaded')
        return filepath
    
    url = f'https://www.gutenberg.org/cache/epub/{gid}/pg{gid}.txt'
    print(f'  {gid}: downloading from {url}')
    resp = requests.get(url)
    resp.raise_for_status()
    
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(resp.text)
    
    time.sleep(2)  # be polite to Gutenberg servers
    return filepath

In [64]:
for gid in CORPUS:
    download_text(gid, source_dir)

  84: downloading from https://www.gutenberg.org/cache/epub/84/pg84.txt
  345: downloading from https://www.gutenberg.org/cache/epub/345/pg345.txt
  42: downloading from https://www.gutenberg.org/cache/epub/42/pg42.txt
  768: downloading from https://www.gutenberg.org/cache/epub/768/pg768.txt
  174: downloading from https://www.gutenberg.org/cache/epub/174/pg174.txt
  696: downloading from https://www.gutenberg.org/cache/epub/696/pg696.txt
  35: downloading from https://www.gutenberg.org/cache/epub/35/pg35.txt
  36: downloading from https://www.gutenberg.org/cache/epub/36/pg36.txt
  159: downloading from https://www.gutenberg.org/cache/epub/159/pg159.txt
  164: downloading from https://www.gutenberg.org/cache/epub/164/pg164.txt
  62: downloading from https://www.gutenberg.org/cache/epub/62/pg62.txt
  139: downloading from https://www.gutenberg.org/cache/epub/139/pg139.txt


# Strip Gutenberg Boilerplate

Remove the header and footer added by Project Gutenberg.

In [65]:
def strip_gutenberg_boilerplate(text):
    """Remove Gutenberg header and footer from plaintext."""
    # Find start of actual text
    start_markers = [
        r'\*\*\*\s*START OF TH(IS|E) PROJECT GUTENBERG EBOOK',
        r'\*\*\*\s*START OF THE PROJECT GUTENBERG',
    ]
    end_markers = [
        r'\*\*\*\s*END OF TH(IS|E) PROJECT GUTENBERG EBOOK',
        r'\*\*\*\s*END OF THE PROJECT GUTENBERG',
        r'End of the Project Gutenberg',
        r'End of Project Gutenberg',
    ]
    
    start_idx = 0
    for pattern in start_markers:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # Move past the marker line
            start_idx = text.index('\n', match.end()) + 1
            break
    
    end_idx = len(text)
    for pattern in end_markers:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            end_idx = match.start()
            break
    
    return text[start_idx:end_idx].strip()

In [66]:
raw_texts = {}
for gid in CORPUS:
    filepath = os.path.join(source_dir, f'{gid}.txt')
    with open(filepath, 'r', encoding='utf-8') as f:
        full_text = f.read()
    raw_texts[gid] = strip_gutenberg_boilerplate(full_text)
    print(f'{gid} ({CORPUS[gid]["title"]}): {len(raw_texts[gid]):,} chars')

84 (Frankenstein): 419,336 chars
345 (Dracula): 845,889 chars
42 (Dr. Jekyll and Mr. Hyde): 139,248 chars
768 (Wuthering Heights): 645,607 chars
174 (The Picture of Dorian Gray): 429,312 chars
696 (The Castle of Otranto): 210,745 chars
35 (The Time Machine): 179,690 chars
36 (The War of the Worlds): 337,423 chars
159 (The Island of Doctor Moreau): 241,161 chars
164 (Twenty Thousand Leagues Under the Sea): 597,840 chars
62 (A Princess of Mars): 371,058 chars
139 (The Lost World): 424,011 chars


# Build LIBRARY Table

Metadata for each source file.

In [67]:
library_rows = []
for gid, meta in CORPUS.items():
    library_rows.append({
        'book_id': gid,
        'title': meta['title'],
        'author': meta['author'],
        'genre': meta['genre'],
        'source_url': f'https://www.gutenberg.org/ebooks/{gid}',
        'format': 'plaintext (UTF-8)',
        'provenance': 'Project Gutenberg',
        'char_count': len(raw_texts[gid]),
    })

LIBRARY = pd.DataFrame(library_rows).set_index('book_id').sort_index()
LIBRARY

,title,author,genre,source_url,format,provenance,char_count
book_id,,,,,,,
35,The Time Machine,"Wells, H. G.",SciFi,https://www.gutenberg.org/ebooks/35,plaintext (UTF-8),Project Gutenberg,179690
36,The War of the Worlds,"Wells, H. G.",SciFi,https://www.gutenberg.org/ebooks/36,plaintext (UTF-8),Project Gutenberg,337423
42,Dr. Jekyll and Mr. Hyde,"Stevenson, Robert Louis",Gothic,https://www.gutenberg.org/ebooks/42,plaintext (UTF-8),Project Gutenberg,139248
62,A Princess of Mars,"Burroughs, Edgar Rice",SciFi,https://www.gutenberg.org/ebooks/62,plaintext (UTF-8),Project Gutenberg,371058
84,Frankenstein,"Shelley, Mary",Gothic,https://www.gutenberg.org/ebooks/84,plaintext (UTF-8),Project Gutenberg,419336
139,The Lost World,"Doyle, Arthur Conan",SciFi,https://www.gutenberg.org/ebooks/139,plaintext (UTF-8),Project Gutenberg,424011
159,The Island of Doctor Moreau,"Wells, H. G.",SciFi,https://www.gutenberg.org/ebooks/159,plaintext (UTF-8),Project Gutenberg,241161
164,Twenty Thousand Leagues Under the Sea,"Verne, Jules",SciFi,https://www.gutenberg.org/ebooks/164,plaintext (UTF-8),Project Gutenberg,597840
174,The Picture of Dorian Gray,"Wilde, Oscar",Gothic,https://www.gutenberg.org/ebooks/174,plaintext (UTF-8),Project Gutenberg,429312


# Parse into Chapters and Paragraphs (F1 — MLCF)

Split each novel into chapters, then paragraphs (separated by blank lines), then sentences.

In [68]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

In [69]:
def split_into_chapters(text):
    """
    Split text into chapters using multiple heading patterns.
    Tries several common Gutenberg conventions in order.
    Returns a list of (chapter_num, chapter_text) tuples.
    """
    patterns = [
        # 'CHAPTER I', 'Chapter 12', 'CHAPTER XXIV', etc.
        re.compile(r'(?:^|\n)\s*(?:CHAPTER|Chapter)\s+(?:[IVXLCDM]+|\d+)[.\s]*(?:\n|$)', re.MULTILINE),
        # Roman numerals alone on a line: 'I.', 'II.', 'XIV.'
        re.compile(r'(?:^|\n)\s*([IVXLCDM]+)\.\s*(?:\n|$)', re.MULTILINE),
        # All-caps titled sections (e.g., Jekyll & Hyde: 'STORY OF THE DOOR')
        re.compile(r'(?:^|\n)\s*([A-Z][A-Z ]{5,})\s*(?:\n|$)', re.MULTILINE),
    ]
    
    for pattern in patterns:
        splits = list(pattern.finditer(text))
        if len(splits) >= 3:  # need at least 3 splits to be a real chapter structure
            chapters = []
            for i, match in enumerate(splits):
                start = match.end()
                end = splits[i + 1].start() if i + 1 < len(splits) else len(text)
                chapter_text = text[start:end].strip()
                if len(chapter_text) > 100:
                    chapters.append((len(chapters) + 1, chapter_text))
            if len(chapters) >= 3:
                return chapters
    
    # Fallback: treat entire text as one chapter
    return [(1, text)]

In [70]:
def parse_book_to_mlcf(gid, text):
    """
    Parse a book's text into MLCF rows:
    (book_id, chapter, para_num, sent_num, sent_str)
    """
    chapters = split_into_chapters(text)
    rows = []
    
    for chap_num, chap_text in chapters:
        # Split into paragraphs by blank lines
        paragraphs = re.split(r'\n\s*\n', chap_text)
        
        for para_num, para in enumerate(paragraphs):
            para = para.strip()
            # Collapse internal newlines into spaces
            para = re.sub(r'\s+', ' ', para)
            if len(para) < 5:
                continue
            
            sentences = sent_tokenize(para)
            for sent_num, sent in enumerate(sentences):
                sent = sent.strip()
                if len(sent) > 0:
                    rows.append({
                        'book_id': gid,
                        'chapter': chap_num,
                        'para_num': para_num,
                        'sent_num': sent_num,
                        'sent_str': sent,
                    })
    
    return rows

In [71]:
all_rows = []
for gid, text in raw_texts.items():
    book_rows = parse_book_to_mlcf(gid, text)
    print(f'{gid} ({CORPUS[gid]["title"]}): {len(book_rows):,} sentences')
    all_rows.extend(book_rows)

84 (Frankenstein): 3,323 sentences
345 (Dracula): 8,721 sentences
42 (Dr. Jekyll and Mr. Hyde): 1,114 sentences
768 (Wuthering Heights): 6,271 sentences
174 (The Picture of Dorian Gray): 6,311 sentences
696 (The Castle of Otranto): 1,967 sentences
35 (The Time Machine): 1,896 sentences
36 (The War of the Worlds): 3,234 sentences
159 (The Island of Doctor Moreau): 2,743 sentences
164 (Twenty Thousand Leagues Under the Sea): 6,528 sentences
62 (A Princess of Mars): 2,328 sentences
139 (The Lost World): 4,473 sentences


In [72]:
CORPUS_DF = pd.DataFrame(all_rows).set_index(OHCO).sort_index()
print(f'Total sentences: {len(CORPUS_DF):,}')
CORPUS_DF.head(10)

Total sentences: 48,909


sent_str
book_id chapter para_num sent_num                                                   
35      1       0        0                                              Introduction
                1        0         The Time Traveller (for so it will be convenie...
                         1         His pale grey eyes shone and twinkled, and his...
                         2         The fire burnt brightly, and the soft radiance...
                         3         Our chairs, being his patents, embraced and ca...
                         4         And he put it to us in this way—marking the po...
                2        0                            “You must follow me carefully.
                         1         I shall have to controvert one or two ideas th...
                         2         The geometry, for instance, they taught you at...
                3        0         “Is not that rather a large thing to expect us...

In [73]:
CORPUS_DF.sample(10)

,,,,sent_str
book_id,chapter,para_num,sent_num,
36,14,26,4,People in fashionable clothing peeped at them ...
696,2,117,0,“Heavens!
62,19,12,1,They had been searching among the northern hor...
345,19,16,0,There were only twenty-nine left out of the fi...
36,25,3,0,There was black dust along the roadway from th...
345,18,61,0,"“He seems very importunate, sir."
768,32,0,2,They’re allas three wick’ after other folk wi’...
35,2,5,2,This saddle represents the seat of a time trav...
174,19,69,1,Life had suddenly become too hideous a burden ...


## Quick Sanity Checks

In [74]:
# Sentences per book
CORPUS_DF.groupby('book_id').size()

book_id
35     1896
36     3234
42     1114
62     2328
84     3323
139    4473
159    2743
164    6528
174    6311
345    8721
696    1967
768    6271
dtype: int64

In [75]:
# Chapters per book
CORPUS_DF.reset_index().groupby('book_id')['chapter'].nunique()

book_id
35     16
36     27
42      6
62     28
84     25
139    16
159    22
164    46
174    21
345    27
696     5
768    34
Name: chapter, dtype: int64

In [76]:
# Average sentence length (chars)
CORPUS_DF['sent_str'].str.len().describe()

count    48909.000000
mean        96.785929
std         75.820801
min          1.000000
25%         42.000000
50%         78.000000
75%        132.000000
max       2541.000000
Name: sent_str, dtype: float64

# Save Outputs

In [77]:
LIBRARY.to_csv(os.path.join(output_dir, 'LIBRARY.csv'))
print('Saved LIBRARY.csv')

Saved LIBRARY.csv


In [78]:
CORPUS_DF.to_csv(os.path.join(output_dir, 'CORPUS_F1.csv'))
print('Saved CORPUS_F1.csv (MLCF)')

Saved CORPUS_F1.csv (MLCF)


# Summary

- Downloaded 12 novels (6 Gothic/Horror, 6 Sci-Fi) from Project Gutenberg.
- Stripped Gutenberg boilerplate headers/footers.
- Parsed into MLCF format: `(book_id, chapter, para_num, sent_num)` → `sent_str`.
- Saved `LIBRARY.csv` and `CORPUS_F1.csv`.

**Next:** Step 2 — Tokenization & STADM (F2).